In [178]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import sklearn
from sklearn.model_selection import train_test_split

![image.png](attachment:image.png)

**FORWARD**:

![image.png](attachment:image.png)

**BACKWARD**:

![image.png](attachment:image.png)

*шаблоны для s1, s2, t1, t2*

In [179]:
# базовый шаблон
class st_net(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]), #
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], output_dim),
            nn.Tanh()
        )

    def forward(self, input):
        return self.net(input)

# линейное преобразование
class LinearNet(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]), #
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.Linear(hidden_dims[1], output_dim),
        )

    def forward(self, input):
        return self.net(input)

In [180]:
# собственно, сама сеть
class INN(nn.Module):
    def __init__(self, input_dim1, input_dim2,
                hidden_dims_s1 = [64, 64], hidden_dims_s2 = [64, 64],
                hidden_dims_t1 = [64, 64], hidden_dims_t2 = [64, 64]):

        super().__init__()
        self.input = None
        self.input_dim1 = input_dim1
        self.input_dim2 = input_dim2

        self.u1 = None
        self.u2 = None
        self.v1 = None
        self.v2 = None

        self.s1_net = st_net(input_dim2, hidden_dims_s1, input_dim1)
        self.s2_net = st_net(input_dim1, hidden_dims_s2, input_dim2)
        self.t1_net = st_net(input_dim2, hidden_dims_t1, input_dim1)
        self.t2_net = st_net(input_dim1, hidden_dims_t2, input_dim2)


    def forward(self, input, reverse=False):
        if not reverse:  # Прямое преобразование

            self.input = input
            self.u1 = self.input[:, :self.input_dim1]
            self.u2 = self.input[:, self.input_dim1:]

            s1 = self.s1_net(self.u2)
            t1 = self.t1_net(self.u2)
            v1 = self.u1 * torch.exp(s1) + t1

            s2 = self.s2_net(v1)
            t2 = self.t2_net(v1)
            v2 = self.u2 * torch.exp(s2) + t2

            self.v1 = v1
            self.v2 = v2
            output = torch.cat((v1, v2), dim=1)

            return output

        else: # Обратный проход

            s2 = self.s2_net(self.v1)
            t2 = self.t2_net(self.v1)
            u2 = (self.v2 - t2) / torch.exp(s2)

            s1 = self.s1_net(u2)
            t1 = self.t1_net(u2)
            u1 = (self.v1 - t1) / torch.exp(s1)

            rerv_input = torch.cat((u1, u2), dim=1)

            return rerv_input

Проверим, что сеть "обращается"

In [181]:
def test_invertible_network():
    batch_size = 3
    model = INN(input_dim1=3, input_dim2=7)
    x = torch.randn(batch_size, 10)
    z = model(x)
    x_rev = model(z, reverse=True)
    print(f" x_fwd ={x}, \n x_rev ={x_rev}, \n z = {z}")
    reconstruction_error = F.mse_loss(x, x_rev)

    return model, reconstruction_error

In [182]:
model, reconstruction_error = test_invertible_network()
print(f"Reconstruction error: {reconstruction_error}")


 x_fwd =tensor([[ 0.7287, -0.1474, -1.2106, -0.1585,  0.1833, -0.4048, -0.6419, -0.9373,
          0.0165,  0.8387],
        [ 0.4693,  0.5550, -0.3767, -0.4023, -0.6372, -0.7080,  0.2543, -0.5572,
         -0.8149, -0.3420],
        [-0.2654, -0.0529,  0.6533,  0.4649,  1.4529, -1.6366, -0.0665,  1.2047,
         -0.0090,  3.0220]]), 
 x_rev =tensor([[ 0.7287, -0.1474, -1.2106, -0.1585,  0.1833, -0.4048, -0.6419, -0.9373,
          0.0165,  0.8387],
        [ 0.4693,  0.5550, -0.3767, -0.4023, -0.6372, -0.7080,  0.2543, -0.5572,
         -0.8149, -0.3420],
        [-0.2654, -0.0529,  0.6533,  0.4649,  1.4529, -1.6366, -0.0665,  1.2047,
         -0.0090,  3.0220]], grad_fn=<CatBackward0>), 
 z = tensor([[ 0.8955, -0.0408, -1.2730, -0.2125,  0.1689, -0.1660, -0.4965, -0.4772,
          0.3025,  0.7806],
        [ 0.8113,  0.6336, -0.4966, -0.4318, -0.5171, -0.7718,  0.4898, -0.2375,
         -0.7853, -0.1785],
        [ 0.1145, -0.3364,  0.3250,  0.4365,  1.2498, -1.3681,  0.2341,  1.40

In [183]:
print(model)

INN(
  (s1_net): LinearNet(
    (net): Sequential(
      (0): Linear(in_features=7, out_features=64, bias=True)
      (1): Linear(in_features=64, out_features=64, bias=True)
      (2): Linear(in_features=64, out_features=3, bias=True)
    )
  )
  (s2_net): LinearNet(
    (net): Sequential(
      (0): Linear(in_features=3, out_features=64, bias=True)
      (1): Linear(in_features=64, out_features=64, bias=True)
      (2): Linear(in_features=64, out_features=7, bias=True)
    )
  )
  (t1_net): LinearNet(
    (net): Sequential(
      (0): Linear(in_features=7, out_features=64, bias=True)
      (1): Linear(in_features=64, out_features=64, bias=True)
      (2): Linear(in_features=64, out_features=3, bias=True)
    )
  )
  (t2_net): LinearNet(
    (net): Sequential(
      (0): Linear(in_features=3, out_features=64, bias=True)
      (1): Linear(in_features=64, out_features=64, bias=True)
      (2): Linear(in_features=64, out_features=7, bias=True)
    )
  )
)


Тест на `"Искуственных данных"`

In [184]:
class SimpleDataset(torch.utils.data.Dataset): #clearing noise example
    def __init__(self, data, a, b):
        self.input = data
        self.targets = a * data + torch.randn(data.shape) * 0.1  + b
        self.labels  = a * data  + b
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.labels[idx], self.targets[idx]

    def show_input(self):
        return self.input

In [185]:
line = torch.arange(-100, 100, 0.01)
data_rows = []
for _ in range(10):
    shuffled_indices = torch.randperm(line.shape[0])
    data_rows.append(line[shuffled_indices])
Data = torch.stack(data_rows)
Data = Data.T
Data

tensor([[ -9.1100,  41.5600,  54.1900,  ..., -28.2500,  12.6000,  26.1200],
        [ 83.0000,   2.1900,  88.5800,  ...,  32.4600,  20.4200, -44.1500],
        [ 64.8000,  85.3400, -18.6400,  ...,  55.8100, -70.5700,  41.6300],
        ...,
        [ 35.4300,   3.5900, -31.1100,  ..., -14.6100,  -3.5700,  18.1500],
        [ 84.1400,  21.6600,  -6.3300,  ...,  98.1100, -34.7300,  85.1800],
        [ 12.8000,  43.6600,   8.2600,  ...,  85.7900,  66.0900,  74.4600]])

In [186]:
a = torch.tensor([0.3, 0.3, 0.3, 0.3,
                  -0.3, -0.3, -0.3, -0.3, -0.3, -0.3])
b = torch.tensor([0.95, 0.95, 0.95, 0.95,
                  -0.95, -0.95, -0.95, -0.95, -0.95, -0.95])

train_data, val_test_data = train_test_split(Data, test_size=0.2, shuffle=True)
val_data, test_data = train_test_split(Data, test_size=0.2)

train_dataset = SimpleDataset(train_data, a, b)
val_dataset   = SimpleDataset(val_data, a, b)[0]
test_dataset  = SimpleDataset(test_data, a, b)[0]

train_dataset[0], train_dataset.show_input()[0]

((tensor([ 20.7680,  20.2340,  11.8520, -25.7260,  -9.4790, -22.0970,  24.1750,
           14.1790, -21.0680,  19.9000]),
  tensor([ 20.7153,  20.0620,  11.7580, -25.7364,  -9.6636, -22.1459,  24.2579,
           14.1868, -20.9923,  19.7063])),
 tensor([ 66.0600,  64.2800,  36.3400, -88.9200,  28.4300,  70.4900, -83.7500,
         -50.4300,  67.0600, -69.5000]))

In [ ]:
#  Тут хватит линейных слоев
class LinearINN(nn.Module):
    def __init__(self, input_dim1, input_dim2,
                hidden_dims_s1 = [64, 64], hidden_dims_s2 = [64, 64],
                hidden_dims_t1 = [64, 64], hidden_dims_t2 = [64, 64]):

        super().__init__()
        self.input = None
        self.input_dim1 = input_dim1
        self.input_dim2 = input_dim2

        self.u1 = None
        self.u2 = None
        self.v1 = None
        self.v2 = None

        self.s1_net = LinearNet(input_dim2, hidden_dims_s1, input_dim1)
        self.s2_net = LinearNet(input_dim1, hidden_dims_s2, input_dim2)
        self.t1_net = LinearNet(input_dim2, hidden_dims_t1, input_dim1)
        self.t2_net = LinearNet(input_dim1, hidden_dims_t2, input_dim2)


    def forward(self, input, reverse=False):
        if not reverse:  # Прямое преобразование

            self.input = input
            self.u1 = self.input[:, :self.input_dim1]
            self.u2 = self.input[:, self.input_dim1:]

            s1 = self.s1_net(self.u2)
            t1 = self.t1_net(self.u2)
            v1 = self.u1 * torch.exp(s1) + t1

            s2 = self.s2_net(v1)
            t2 = self.t2_net(v1)
            v2 = self.u2 * torch.exp(s2) + t2

            self.v1 = v1
            self.v2 = v2
            output = torch.cat((v1, v2), dim=1)

            return output

        else: # Обратный проход

            s2 = self.s2_net(self.v1)
            t2 = self.t2_net(self.v1)
            u2 = (self.v2 - t2) / torch.exp(s2)

            s1 = self.s1_net(u2)
            t1 = self.t1_net(u2)
            u1 = (self.v1 - t1) / torch.exp(s1)

            rerv_input = torch.cat((u1, u2), dim=1)

            return rerv_input

In [187]:
model = LinearINN(input_dim1=4, input_dim2=6)